In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/sample_submission.csv
/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv
/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv
/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv


In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv")
sub_df = pd.read_csv("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/sample_submission.csv")
metadata = pd.read_csv("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv")

In [3]:
train_df.head()

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221 entries, 0 to 220
Data columns (total 37 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   event_id                      221 non-null    int64  
 1   num_perimeters_0_5h           221 non-null    int64  
 2   dt_first_last_0_5h            221 non-null    float64
 3   low_temporal_resolution_0_5h  221 non-null    int64  
 4   area_first_ha                 221 non-null    float64
 5   area_growth_abs_0_5h          221 non-null    float64
 6   area_growth_rel_0_5h          221 non-null    float64
 7   area_growth_rate_ha_per_h     221 non-null    float64
 8   log1p_area_first              221 non-null    float64
 9   log1p_growth                  221 non-null    float64
 10  log_area_ratio_0_5h           221 non-null    float64
 11  relative_growth_0_5h          221 non-null    float64
 12  radial_growth_m               221 non-null    float64
 13  radia

In [5]:
train_df['event'].value_counts()

event
0    152
1     69
Name: count, dtype: int64

In [6]:
test_df.head()

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,projected_advance_m,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month
0,10662602,1,0.000000,1,2.452217,0.000000,0.00000,0.000000,1.239017,0.000000,...,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0,3,7
1,13353600,1,0.000000,1,131.669588,0.000000,0.00000,0.000000,4.887862,0.000000,...,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,22,0,8
2,13942327,1,0.000000,1,6.723104,0.000000,0.00000,0.000000,2.044216,0.000000,...,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,2,6,7
3,16112781,1,0.000000,1,285.416736,0.000000,0.00000,0.000000,5.657448,0.000000,...,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0,1,7
4,17132808,7,3.459331,0,61.098604,12.516633,0.20486,3.618224,4.128724,2.603921,...,13.54413,-22.687575,0.044572,0.15855,0.15855,-24.414806,3.920562,23,5,7


In [7]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 35 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   event_id                      95 non-null     int64  
 1   num_perimeters_0_5h           95 non-null     int64  
 2   dt_first_last_0_5h            95 non-null     float64
 3   low_temporal_resolution_0_5h  95 non-null     int64  
 4   area_first_ha                 95 non-null     float64
 5   area_growth_abs_0_5h          95 non-null     float64
 6   area_growth_rel_0_5h          95 non-null     float64
 7   area_growth_rate_ha_per_h     95 non-null     float64
 8   log1p_area_first              95 non-null     float64
 9   log1p_growth                  95 non-null     float64
 10  log_area_ratio_0_5h           95 non-null     float64
 11  relative_growth_0_5h          95 non-null     float64
 12  radial_growth_m               95 non-null     float64
 13  radial_

In [8]:
sub_df.head()

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.5,0.5,0.5,0.5
1,13353600,0.5,0.5,0.5,0.5
2,13942327,0.5,0.5,0.5,0.5
3,16112781,0.5,0.5,0.5,0.5
4,17132808,0.5,0.5,0.5,0.5


In [9]:
metadata

,column,type,category,description,units,range
0,event_id,identifier,identifier,Anonymized fire event identifier (stable rando...,NaN,NaN
1,time_to_hit_hours,target,target,Time from t0+5h until fire comes within 5km of...,hours,"[0, 72]"
2,event,target,target,"Event indicator: 1 if fire hit within 72h, 0 i...",NaN,NaN
3,num_perimeters_0_5h,feature,temporal_coverage,Number of perimeters within first 5 hours,NaN,NaN
4,dt_first_last_0_5h,feature,temporal_coverage,Time span between first and last perimeter (ho...,NaN,NaN
5,low_temporal_resolution_0_5h,feature,temporal_coverage,"Flag: 1 if dt < 0.5h or only 1 perimeter, else 0",NaN,NaN
6,area_first_ha,feature,growth,Initial fire area at t0 (hectares),NaN,NaN
7,area_growth_abs_0_5h,feature,growth,Feature from growth category,NaN,NaN
8,area_growth_rel_0_5h,feature,growth,Feature from growth category,NaN,NaN
9,area_growth_rate_ha_per_h,feature,growth,Area growth rate (hectares per hour),NaN,NaN


In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 2. Define Horizons and Features
horizons = [12, 24, 48, 72]
features = [col for col in test_df.columns if col not in ['event_id']]
X = train_df[features]

# 3. Validation Phase (Metrics)
print("--- Validation Metrics ---")
for h in horizons:
    # Create the binary target for this specific horizon
    y = ((train_df['event'] == 1) & (train_df['time_to_hit_hours'] <= h)).astype(int)
    
    # Split for metric evaluation
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Calculate scores on the validation set
    probs = model.predict_proba(X_val)[:, 1]
    preds = model.predict(X_val)
    
    print(f"Horizon {h}h | Accuracy: {accuracy_score(y_val, preds):.4f} | ROC-AUC: {roc_auc_score(y_val, probs):.4f}")

--- Validation Metrics ---
Horizon 12h | Accuracy: 0.9111 | ROC-AUC: 0.9811
Horizon 24h | Accuracy: 0.9778 | ROC-AUC: 0.9698
Horizon 48h | Accuracy: 0.9778 | ROC-AUC: 1.0000
Horizon 72h | Accuracy: 1.0000 | ROC-AUC: 1.0000


In [11]:
# 4. Final Prediction Phase (Submission)
results = {'event_id': test_df['event_id']}
for h in horizons:
    y = ((train_df['event'] == 1) & (train_df['time_to_hit_hours'] <= h)).astype(int)
    
    # Train on FULL training data for maximum accuracy
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)
    
    results[f'{h}h'] = model.predict_proba(test_df[features])[:, 1]

# 5. Save and enforce logical monotonicity
submission = pd.DataFrame(results)
cols = ['12h', '24h', '48h', '72h']
# P(12h) cannot be higher than P(24h), etc.
submission[cols] = np.maximum.accumulate(submission[cols].values, axis=1)

submission.head()

,event_id,12h,24h,48h,72h
0,10662602,0.02,0.07,0.09,0.13
1,13353600,0.36,0.83,0.83,0.83
2,13942327,0.01,0.01,0.03,0.03
3,16112781,0.76,0.89,0.92,0.92
4,17132808,0.67,0.67,0.67,0.67


In [12]:
submission.to_csv('submission.csv', index=False)